## Step 2: Transformation — Clean, Enrich, Join

**Project:** AI-Powered Sales Data Insights with Snowflake + Cortex Analyst

**Architecture:** Ingestion → TRANSFORMATION → Delivery → Observability

**What this notebook does:**
1. Fix the product key mismatch found in Step 1 (keys 310-351 missing from DIM_PRODUCT)
2. Clean and standardize all dimension tables using **pandas on Snowflake**
3. Enrich customer data with calculated fields (age, income bands)
4. Build the **star-schema join** (Sales × Product × Customer × Geography)
5. Create aggregation tables (product performance, territory, daily trend)
6. Save everything to `CURATED` schema

Following [DE_100 Notebook](https://quickstarts.snowflake.com/guide/intro-to-data-engineering-python/) patterns:
- `modin.pandas` (pandas on Snowflake) for data cleaning
- `Snowpark DataFrame` API for aggregations
- `save_as_table()` for persistence

In [ ]:
# Snowpark Pandas API (same as DE_100)
import modin.pandas as spd
import snowflake.snowpark.modin.plugin

# Snowpark DataFrame API
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import StructType, StructField, StringType, FloatType, IntegerType, DateType
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()
session.query_tag = {
    "origin": "raj_manala",
    "name": "sales_analytics_ai_insights",
    "version": {"major": 1, "minor": 0},
    "attributes": {"step": "transformation"}
}
print(f"Session ready: {session.get_current_database()}")

In [ ]:
USE DATABASE SALES_ANALYTICS_2025;
USE SCHEMA RAW_DATA;

### TRANSFORM 1: Fix the Product Key Mismatch

In Step 1 we found that `FACT_INTERNET_SALES` uses product keys **310-351** (finished bikes), but `DIM_PRODUCT` only has keys **1-100** (raw components). 

We need to create an enriched product lookup table with the correct metadata for the finished goods that actually appear in our sales data.

In [ ]:
-- Create the enriched product lookup for finished goods (keys 310-351)
-- These are the actual bikes sold in FACT_INTERNET_SALES
USE SCHEMA CURATED;

CREATE OR REPLACE TABLE DIM_PRODUCT_ENRICHED (
    product_key INT,
    product_name VARCHAR,
    model VARCHAR,
    product_line VARCHAR,
    subcategory VARCHAR,
    category VARCHAR,
    color VARCHAR,
    size VARCHAR,
    list_price FLOAT,
    standard_cost FLOAT
);

INSERT INTO DIM_PRODUCT_ENRICHED VALUES
(310, 'Mountain-200 Black, 38',   'Mountain-200', 'Mountain', 'Mountain Bikes', 'Bikes', 'Black',  '38', 2294.99, 1251.98),
(311, 'Mountain-200 Black, 42',   'Mountain-200', 'Mountain', 'Mountain Bikes', 'Bikes', 'Black',  '42', 2294.99, 1251.98),
(312, 'Mountain-200 Black, 46',   'Mountain-200', 'Mountain', 'Mountain Bikes', 'Bikes', 'Black',  '46', 2294.99, 1251.98),
(313, 'Mountain-200 Silver, 38',  'Mountain-200', 'Mountain', 'Mountain Bikes', 'Bikes', 'Silver', '38', 2319.99, 1265.62),
(314, 'Mountain-200 Silver, 46',  'Mountain-200', 'Mountain', 'Mountain Bikes', 'Bikes', 'Silver', '46', 2319.99, 1265.62),
(326, 'Road-650 Black, 44',      'Road-650',     'Road',     'Road Bikes',     'Bikes', 'Black',  '44', 782.99,  461.50),
(330, 'Road-650 Black, 52',      'Road-650',     'Road',     'Road Bikes',     'Bikes', 'Black',  '52', 782.99,  461.50),
(332, 'Road-650 Red, 44',        'Road-650',     'Road',     'Road Bikes',     'Bikes', 'Red',    '44', 782.99,  461.50),
(336, 'Road-650 Red, 52',        'Road-650',     'Road',     'Road Bikes',     'Bikes', 'Red',    '52', 782.99,  461.50),
(342, 'Road-650 Red, 62',        'Road-650',     'Road',     'Road Bikes',     'Bikes', 'Red',    '62', 782.99,  461.50),
(344, 'Road-350-W Yellow, 40',   'Road-350-W',   'Road',     'Road Bikes',     'Bikes', 'Yellow', '40', 1700.99, 1015.73),
(345, 'Road-350-W Yellow, 42',   'Road-350-W',   'Road',     'Road Bikes',     'Bikes', 'Yellow', '42', 1700.99, 1015.73),
(346, 'Road-350-W Yellow, 44',   'Road-350-W',   'Road',     'Road Bikes',     'Bikes', 'Yellow', '44', 1700.99, 1015.73),
(347, 'Road-350-W Yellow, 48',   'Road-350-W',   'Road',     'Road Bikes',     'Bikes', 'Yellow', '48', 1700.99, 1015.73),
(350, 'Road-150 Red, 44',        'Road-150',     'Road',     'Road Bikes',     'Bikes', 'Red',    '44', 3578.27, 2171.29),
(351, 'Road-150 Red, 48',        'Road-150',     'Road',     'Road Bikes',     'Bikes', 'Red',    '48', 3578.27, 2171.29);

SELECT * FROM DIM_PRODUCT_ENRICHED ORDER BY product_key;

In [ ]:
-- Verify: ALL product keys in sales now have a match
SELECT 
    COUNT(DISTINCT f.PRODUCTKEY) AS sales_products,
    COUNT(DISTINCT p.product_key) AS matched_products,
    COUNT(DISTINCT CASE WHEN p.product_key IS NULL THEN f.PRODUCTKEY END) AS still_unmatched
FROM SALES_ANALYTICS_2025.RAW_DATA.FACT_INTERNET_SALES f
LEFT JOIN SALES_ANALYTICS_2025.CURATED.DIM_PRODUCT_ENRICHED p 
    ON f.PRODUCTKEY = p.product_key;

### TRANSFORM 2: Clean & Enrich DIM_CUSTOMER

Following the DE_100 notebook pattern:
- Rename columns to snake_case (like `rename_columns` cell in DE_100)
- Clean data types
- Add calculated fields: `age`, `income_band`, `age_group`, `full_name`

In [ ]:
# Load raw customer data using pandas on Snowflake (same as DE_100)
customer_mdf = spd.read_snowflake('SALES_ANALYTICS_2025.RAW_DATA.DIM_CUSTOMER')
print(f"Raw DIM_CUSTOMER: {len(customer_mdf)} rows × {customer_mdf.shape[1]} cols")
customer_mdf.head(3)

In [ ]:
# Rename columns to snake_case (same pattern as DE_100 rename_columns cell)
customer_clean = customer_mdf.rename(columns={
    'CUSTOMERKEY': 'customer_key',
    'GEOGRAPHYKEY': 'geography_key',
    'FIRSTNAME': 'first_name',
    'LASTNAME': 'last_name',
    'BIRTHDATE': 'birth_date',
    'MARITALSTATUS': 'marital_status',
    'GENDER': 'gender',
    'EMAILADDRESS': 'email',
    'YEARLYINCOME': 'yearly_income',
    'TOTALCHILDREN': 'total_children',
    'NUMBERCHILDRENATHOME': 'children_at_home',
    'ENGLISHEDUCATION': 'education',
    'ENGLISHOCCUPATION': 'occupation',
    'HOUSEOWNERFLAG': 'house_owner',
    'NUMBERCARSOWNED': 'cars_owned',
    'DATEFIRSTPURCHASE': 'first_purchase_date',
    'COMMUTEDISTANCE': 'commute_distance'
})

# Select only the columns we need
customer_clean = customer_clean[[
    'customer_key', 'geography_key', 'first_name', 'last_name',
    'birth_date', 'marital_status', 'gender', 'email', 'yearly_income',
    'total_children', 'children_at_home', 'education', 'occupation',
    'house_owner', 'cars_owned', 'first_purchase_date', 'commute_distance'
]]

print(f"Cleaned columns: {list(customer_clean.columns)}")

In [ ]:
# Add calculated fields
# full_name
customer_clean['full_name'] = customer_clean['first_name'] + ' ' + customer_clean['last_name']

# income_band (similar to DE_100 clean_price pattern — apply function to column)
def get_income_band(income):
    try:
        income = float(income)
        if income < 30000: return 'Low (<$30K)'
        elif income < 60000: return 'Medium ($30K-$60K)'
        elif income < 100000: return 'High ($60K-$100K)'
        else: return 'Very High ($100K+)'
    except:
        return 'Unknown'

customer_clean['income_band'] = customer_clean['yearly_income'].apply(get_income_band)

print("\nIncome band distribution:")
print(customer_clean['income_band'].value_counts())

In [ ]:
# Preview the enriched customer data
print(f"\nEnriched DIM_CUSTOMER: {len(customer_clean)} rows × {customer_clean.shape[1]} cols")
print(f"New columns: full_name, income_band")
customer_clean[['customer_key', 'full_name', 'gender', 'yearly_income', 'income_band', 'education', 'occupation']].head(10)

### TRANSFORM 3: Build the Star-Schema Join

This is the core transformation — join FACT_INTERNET_SALES with all dimension tables to create one fully enriched, denormalized fact table.

We use **Snowpark DataFrame** API for this (same pattern as DE_100 `join_tables` cell).

```
FACT_INTERNET_SALES
    × DIM_PRODUCT_ENRICHED  (on PRODUCTKEY)
    × DIM_CUSTOMER          (on CUSTOMERKEY) 
    × Territory mapping     (on SALESTERRITORYKEY)
```

Plus: calculated fields for `gross_profit`, `profit_margin_pct`, `shipping_days`.

In [ ]:
-- Build the fully enriched fact table using SQL
-- This is the STAR-SCHEMA JOIN: Sales × Product × Customer × Territory
USE SCHEMA CURATED;

CREATE OR REPLACE TABLE FACT_SALES_ENRICHED AS
SELECT 
    -- Order fields
    f.SALESORDERNUMBER                    AS order_number,
    f.SALESORDERLINENUMBER                AS line_number,
    f.ORDERQUANTITY                       AS order_quantity,
    f.UNITPRICE                           AS unit_price,
    f.SALESAMOUNT                         AS sales_amount,
    f.PRODUCTSTANDARDCOST                 AS standard_cost,
    f.TOTALPRODUCTCOST                    AS total_cost,
    
    -- Calculated: Profitability
    (f.SALESAMOUNT - f.TOTALPRODUCTCOST)  AS gross_profit,
    ROUND((f.SALESAMOUNT - f.TOTALPRODUCTCOST) / NULLIF(f.SALESAMOUNT, 0) * 100, 2) 
                                          AS profit_margin_pct,
    
    -- Calculated: Costs
    f.UNITPRICEDISCOUNTPCT                AS discount_pct,
    f.DISCOUNTAMOUNT                      AS discount_amount,
    f.TAXAMT                              AS tax_amount,
    f.FREIGHT                             AS freight,
    (f.SALESAMOUNT + f.TAXAMT)            AS total_with_tax,
    
    -- Dates
    f.ORDERDATE                           AS order_date,
    f.SHIPDATE                            AS ship_date,
    f.DUEDATE                             AS due_date,
    DATEDIFF('day', f.ORDERDATE, f.SHIPDATE) AS shipping_days,
    DAYNAME(f.ORDERDATE)                  AS order_day_of_week,
    
    -- Product dimension (from our enriched lookup)
    p.product_key,
    p.product_name,
    p.model,
    p.product_line,
    p.subcategory,
    p.category,
    p.color,
    p.size                                AS product_size,
    
    -- Customer dimension
    f.CUSTOMERKEY                         AS customer_key,
    c.FIRSTNAME || ' ' || c.LASTNAME      AS full_name,
    c.GENDER                              AS gender,
    c.YEARLYINCOME                        AS yearly_income,
    c.ENGLISHEDUCATION                    AS education,
    c.ENGLISHOCCUPATION                   AS occupation,
    c.MARITALSTATUS                       AS marital_status,
    c.COMMUTEDISTANCE                     AS commute_distance,
    CASE 
        WHEN c.YEARLYINCOME < 30000  THEN 'Low (<$30K)'
        WHEN c.YEARLYINCOME < 60000  THEN 'Medium ($30K-$60K)'
        WHEN c.YEARLYINCOME < 100000 THEN 'High ($60K-$100K)'
        ELSE 'Very High ($100K+)'
    END                                   AS income_band,
    
    -- Territory (derived from sales territory key)
    f.SALESTERRITORYKEY                   AS territory_key,
    CASE f.SALESTERRITORYKEY
        WHEN 1  THEN 'Northwest'
        WHEN 2  THEN 'Northeast'
        WHEN 3  THEN 'Central'
        WHEN 4  THEN 'Southwest'
        WHEN 5  THEN 'Southeast'
        WHEN 6  THEN 'Canada'
        WHEN 7  THEN 'France'
        WHEN 8  THEN 'Germany'
        WHEN 9  THEN 'Australia'
        WHEN 10 THEN 'United Kingdom'
    END                                   AS territory_name,
    
    -- Geography (via customer)
    g.CITY                                AS city,
    g.STATEPROVINCENAME                   AS state_province,
    g.ENGLISHCOUNTRYREGIONNAME            AS country

FROM SALES_ANALYTICS_2025.RAW_DATA.FACT_INTERNET_SALES f

-- Join 1: Product enriched lookup
LEFT JOIN SALES_ANALYTICS_2025.CURATED.DIM_PRODUCT_ENRICHED p
    ON f.PRODUCTKEY = p.product_key

-- Join 2: Customer dimension
LEFT JOIN SALES_ANALYTICS_2025.RAW_DATA.DIM_CUSTOMER c
    ON f.CUSTOMERKEY = c.CUSTOMERKEY

-- Join 3: Geography (via customer's geography key)
LEFT JOIN SALES_ANALYTICS_2025.RAW_DATA.DIM_GEOGRAPHY g
    ON c.GEOGRAPHYKEY = g.GEOGRAPHYKEY;

In [ ]:
# Verify the enriched table
enriched_sdf = session.table('SALES_ANALYTICS_2025.CURATED.FACT_SALES_ENRICHED')

print(f"FACT_SALES_ENRICHED: {enriched_sdf.count()} rows × {len(enriched_sdf.schema.fields)} columns")
print(f"\nColumns:")
for field in enriched_sdf.schema.fields:
    print(f"  {field.name:<25} {str(field.datatype)}")

In [ ]:
# Preview the data
enriched_sdf.select(
    'ORDER_NUMBER', 'PRODUCT_NAME', 'MODEL', 'TERRITORY_NAME',
    'SALES_AMOUNT', 'GROSS_PROFIT', 'PROFIT_MARGIN_PCT', 'FULL_NAME'
).show(10)

In [ ]:
-- Data quality check: verify join results
SELECT 
    COUNT(*) AS total_rows,
    COUNT(product_name) AS products_matched,
    COUNT(full_name) AS customers_matched,
    COUNT(territory_name) AS territories_matched,
    COUNT(city) AS geography_matched,
    ROUND(SUM(sales_amount), 2) AS total_revenue,
    ROUND(SUM(gross_profit), 2) AS total_profit,
    ROUND(AVG(profit_margin_pct), 2) AS avg_margin
FROM SALES_ANALYTICS_2025.CURATED.FACT_SALES_ENRICHED;

### TRANSFORM 4: Build Aggregated Analytics Tables

Using **Snowpark DataFrame** API (same pattern as DE_100 `product_order_counts` and `product_status_pivot` cells).

These pre-aggregated tables power the dashboard and give Cortex Analyst faster answers.

In [ ]:
# PRODUCT_PERFORMANCE: Revenue, profit, margin by product
# (same pattern as DE_100 product_order_counts cell)

product_perf = enriched_sdf.group_by('PRODUCT_NAME', 'MODEL', 'PRODUCT_LINE', 'CATEGORY') \
    .agg(
        F.round(F.sum('SALES_AMOUNT'), 2).alias('TOTAL_REVENUE'),
        F.round(F.sum('GROSS_PROFIT'), 2).alias('TOTAL_PROFIT'),
        F.count('*').alias('TOTAL_ORDERS'),
        F.round(F.avg('SALES_AMOUNT'), 2).alias('AVG_ORDER_VALUE'),
        F.round(F.avg('PROFIT_MARGIN_PCT'), 2).alias('AVG_PROFIT_MARGIN'),
        F.round(F.avg('SHIPPING_DAYS'), 1).alias('AVG_SHIPPING_DAYS')
    ) \
    .sort(F.col('TOTAL_REVENUE').desc())

print("\n=== Product Performance ===")
product_perf.show()

# Save to CURATED schema (same as DE_100 save_product_sentiment)
product_perf.write.save_as_table(
    'SALES_ANALYTICS_2025.CURATED.PRODUCT_PERFORMANCE', 
    mode='overwrite'
)
print("✓ Saved CURATED.PRODUCT_PERFORMANCE")

In [ ]:
# TERRITORY_PERFORMANCE: Revenue by sales territory

territory_perf = enriched_sdf.group_by('TERRITORY_NAME') \
    .agg(
        F.round(F.sum('SALES_AMOUNT'), 2).alias('TOTAL_REVENUE'),
        F.round(F.sum('GROSS_PROFIT'), 2).alias('TOTAL_PROFIT'),
        F.count('*').alias('TOTAL_ORDERS'),
        F.round(F.avg('SALES_AMOUNT'), 2).alias('AVG_ORDER_VALUE'),
        F.count_distinct('CUSTOMER_KEY').alias('UNIQUE_CUSTOMERS')
    ) \
    .sort(F.col('TOTAL_REVENUE').desc())

print("\n=== Territory Performance ===")
territory_perf.show()

territory_perf.write.save_as_table(
    'SALES_ANALYTICS_2025.CURATED.TERRITORY_PERFORMANCE',
    mode='overwrite'
)
print("✓ Saved CURATED.TERRITORY_PERFORMANCE")

In [ ]:
# MODEL_COMPARISON: Mountain-200 vs Road models

model_comp = enriched_sdf.group_by('MODEL') \
    .agg(
        F.round(F.sum('SALES_AMOUNT'), 2).alias('TOTAL_REVENUE'),
        F.round(F.sum('GROSS_PROFIT'), 2).alias('TOTAL_PROFIT'),
        F.count('*').alias('TOTAL_ORDERS'),
        F.round(F.avg('UNIT_PRICE'), 2).alias('AVG_UNIT_PRICE'),
        F.round(F.avg('PROFIT_MARGIN_PCT'), 2).alias('AVG_PROFIT_MARGIN')
    ) \
    .sort(F.col('TOTAL_REVENUE').desc())

print("\n=== Model Comparison ===")
model_comp.show()

model_comp.write.save_as_table(
    'SALES_ANALYTICS_2025.CURATED.MODEL_COMPARISON',
    mode='overwrite'
)
print("✓ Saved CURATED.MODEL_COMPARISON")

In [ ]:
# DAILY_SALES_TREND: Time-series revenue and profit

daily_trend = enriched_sdf.group_by(F.to_date('ORDER_DATE').alias('DATE')) \
    .agg(
        F.round(F.sum('SALES_AMOUNT'), 2).alias('REVENUE'),
        F.round(F.sum('GROSS_PROFIT'), 2).alias('PROFIT'),
        F.count('*').alias('ORDERS'),
        F.round(F.avg('SALES_AMOUNT'), 2).alias('AVG_ORDER_VALUE')
    ) \
    .sort(F.col('DATE'))

print("\n=== Daily Sales Trend ===")
daily_trend.show()

daily_trend.write.save_as_table(
    'SALES_ANALYTICS_2025.CURATED.DAILY_SALES_TREND',
    mode='overwrite'
)
print("✓ Saved CURATED.DAILY_SALES_TREND")

In [ ]:
# CUSTOMER_SEGMENTATION: Spending patterns by customer

cust_seg = enriched_sdf.filter(F.col('FULL_NAME').is_not_null()) \
    .group_by('CUSTOMER_KEY', 'FULL_NAME', 'GENDER', 'INCOME_BAND', 'EDUCATION', 'OCCUPATION') \
    .agg(
        F.round(F.sum('SALES_AMOUNT'), 2).alias('TOTAL_SPENT'),
        F.round(F.sum('GROSS_PROFIT'), 2).alias('TOTAL_PROFIT_GENERATED'),
        F.count('*').alias('ORDER_COUNT'),
        F.round(F.avg('SALES_AMOUNT'), 2).alias('AVG_ORDER_VALUE'),
        F.min('ORDER_DATE').alias('FIRST_ORDER'),
        F.max('ORDER_DATE').alias('LAST_ORDER')
    ) \
    .sort(F.col('TOTAL_SPENT').desc())

print("\n=== Customer Segmentation ===")
cust_seg.show()

cust_seg.write.save_as_table(
    'SALES_ANALYTICS_2025.CURATED.CUSTOMER_SEGMENTATION',
    mode='overwrite'
)
print("✓ Saved CURATED.CUSTOMER_SEGMENTATION")

### Copy FACT_BUDGET to CURATED schema

In [ ]:
-- Copy budget to curated schema for Cortex Analyst access
CREATE OR REPLACE TABLE SALES_ANALYTICS_2025.CURATED.BUDGET AS
SELECT * FROM SALES_ANALYTICS_2025.RAW_DATA.FACT_BUDGET;

### Final Validation: All curated tables

In [ ]:
-- Show all curated tables with row counts
SELECT TABLE_NAME, ROW_COUNT, CREATED
FROM SALES_ANALYTICS_2025.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'CURATED'
ORDER BY TABLE_NAME;

In [ ]:
# Final KPI summary from curated data
enriched_sdf = session.table('SALES_ANALYTICS_2025.CURATED.FACT_SALES_ENRICHED')

kpis = enriched_sdf.agg(
    F.round(F.sum('SALES_AMOUNT'), 2).alias('TOTAL_REVENUE'),
    F.round(F.sum('GROSS_PROFIT'), 2).alias('TOTAL_PROFIT'),
    F.round(F.avg('PROFIT_MARGIN_PCT'), 2).alias('AVG_MARGIN'),
    F.count('*').alias('TOTAL_ORDERS'),
    F.round(F.avg('SALES_AMOUNT'), 2).alias('AVG_ORDER_VALUE'),
    F.count_distinct('PRODUCT_KEY').alias('UNIQUE_PRODUCTS'),
    F.count_distinct('TERRITORY_NAME').alias('TERRITORIES'),
    F.round(F.avg('SHIPPING_DAYS'), 1).alias('AVG_SHIPPING_DAYS')
).collect()[0]

print("=" * 55)
print("  STEP 2 COMPLETE: TRANSFORMATION")
print("=" * 55)
print(f"")
print(f"  ┌───────────────────────────────────────────┐")
print(f"  │  CURATED DATA SUMMARY                      │")
print(f"  ├───────────────────────────────────────────┤")
print(f"  │  Total Revenue:    ${kpis['TOTAL_REVENUE']:>12,.2f}       │")
print(f"  │  Total Profit:     ${kpis['TOTAL_PROFIT']:>12,.2f}       │")
print(f"  │  Avg Margin:       {kpis['AVG_MARGIN']:>11.1f}%        │")
print(f"  │  Total Orders:     {kpis['TOTAL_ORDERS']:>12}        │")
print(f"  │  Avg Order Value:  ${kpis['AVG_ORDER_VALUE']:>12,.2f}       │")
print(f"  │  Unique Products:  {kpis['UNIQUE_PRODUCTS']:>12}        │")
print(f"  │  Territories:      {kpis['TERRITORIES']:>12}        │")
print(f"  │  Avg Ship Days:    {kpis['AVG_SHIPPING_DAYS']:>12}        │")
print(f"  └───────────────────────────────────────────┘")
print(f"")
print(f"  Tables saved to CURATED schema:")
print(f"    ✓ FACT_SALES_ENRICHED")
print(f"    ✓ DIM_PRODUCT_ENRICHED")
print(f"    ✓ PRODUCT_PERFORMANCE")
print(f"    ✓ TERRITORY_PERFORMANCE")
print(f"    ✓ MODEL_COMPARISON")
print(f"    ✓ DAILY_SALES_TREND")
print(f"    ✓ CUSTOMER_SEGMENTATION")
print(f"    ✓ BUDGET")
print(f"")
print(f"  ▶ NEXT: Run Step3_DataScience_ML.ipynb")
print(f"    OR skip to Step4_Delivery_CortexAnalyst.ipynb")
print(f"    (Step 3 is optional — Cortex Analyst works without it)")